# Bag-of-Words Retrieval Experiment

This experiment evaluates Bag-of-Words (BoW) as a basic lexical retrieval
method for mapping cyber threat intelligence sentences to MITRE ATT&CK
techniques.

The same MITRE ATT&CK retrieval corpus and TRAM evaluation dataset used in
the TF-IDF experiment are retained to ensure a fair comparison.

Bag-of-Words represents text using word occurrence counts without considering
word order or semantic meaning. Cosine similarity is used to compare each
TRAM threat description against the MITRE ATT&CK technique descriptions.

Performance is evaluated using:
- Recall@1
- Recall@5
- Recall@10
- Mean Reciprocal Rank (MRR)

BoW experiment docs:
- bagofwords_retrieval.ipynb
- bow_summary.csv
- bow_query_results.csv

In [2]:
import pandas as pd

mitre_df = pd.read_csv(
    "mitre_techniques.csv"
)

tram_df = pd.read_csv(
    "tram_single_label_compatible.csv"
)

In [5]:
# same clenaing as tf-idf
import re

def clean_mitre_text(text):
    text = str(text)

    # Remove MITRE citation markers
    text = re.sub(
        r"\(Citation:[^)]+\)",
        "",
        text
    )

    # Convert Markdown links to visible text
    text = re.sub(
        r"\[([^\]]+)\]\([^)]+\)",
        r"\1",
        text
    )

    # Remove backticks but preserve contents
    text = text.replace("`", "")

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [6]:
#combine technique name and description again
mitre_df["retrieval_text"] = (
    mitre_df["name"].fillna("")
    + ". "
    + mitre_df["description"].fillna("")
)

In [7]:
mitre_df["retrieval_text"] = (
    mitre_df["retrieval_text"]
    .apply(clean_mitre_text)
)

In [8]:
#prepare tram data
tram_df["query_text"] = (
    tram_df["sentence"]
    .fillna("")
    .str.strip()
)

In [9]:
#bag of words
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english"
)

In [10]:
#fit
mitre_matrix = vectorizer.fit_transform(
    mitre_df["retrieval_text"]
)

In [11]:
#check
print(mitre_matrix.shape)
print(len(vectorizer.vocabulary_))

(697, 5907)
5907


In [12]:
#transfrom
query_matrix = vectorizer.transform(
    tram_df["query_text"]
)

In [13]:
#test 1 query
query_index = 0

query_text = tram_df.iloc[
    query_index
]["query_text"]

ground_truth = tram_df.iloc[
    query_index
]["technique_id"]

print("QUERY:")
print(query_text)

print("\nGROUND TRUTH:")
print(ground_truth)

QUERY:
It spreads to Microsoft Windows machines using several propagation methods, including the EternalBlue exploit for the CVE-2017-0144 vulnerability in the SMB service.

GROUND TRUTH:
T1210


In [16]:
#calculate cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
scores = cosine_similarity(
    query_matrix[query_index],
    mitre_matrix
).flatten()

In [18]:
#rank
import numpy as np
top_indices = np.argsort(
    scores
)[::-1][:10]

In [19]:
print("Top 10 Bag-of-Words results:\n")

for rank, idx in enumerate(
    top_indices,
    start=1
):
    print(
        f"{rank}. "
        f"{mitre_df.iloc[idx]['technique_id']} - "
        f"{mitre_df.iloc[idx]['name']} "
        f"(score={scores[idx]:.4f})"
    )

Top 10 Bag-of-Words results:

1. T1543.003 - Windows Service (score=0.3326)
2. T1021.002 - SMB/Windows Admin Shares (score=0.2607)
3. T1569.002 - Service Execution (score=0.2422)
4. T1588.006 - Vulnerabilities (score=0.2246)
5. T1496.004 - Cloud Service Hijacking (score=0.2023)
6. T1547.001 - Registry Run Keys / Startup Folder (score=0.1949)
7. T1558.003 - Kerberoasting (score=0.1899)
8. T1021.006 - Windows Remote Management (score=0.1846)
9. T1588.005 - Exploits (score=0.1741)
10. T1587.004 - Exploits (score=0.1673)


In [21]:
#now move on to all
id_to_index = {
    technique_id: idx
    for idx, technique_id
    in enumerate(mitre_df["technique_id"])
}

In [22]:
similarity_matrix = cosine_similarity(
    query_matrix,
    mitre_matrix
)

print(
    "Similarity matrix:",
    similarity_matrix.shape
)

Similarity matrix: (3077, 697)


In [23]:
ranks = []

for i in range(len(tram_df)):

    ground_truth = tram_df.iloc[i][
        "technique_id"
    ]

    correct_index = id_to_index[
        ground_truth
    ]

    scores = similarity_matrix[i]

    ranked_indices = np.argsort(
        scores
    )[::-1]

    correct_rank = (
        np.where(
            ranked_indices == correct_index
        )[0][0]
        + 1
    )

    ranks.append(correct_rank)

In [24]:
tram_df["bow_rank"] = ranks
tram_df[
    [
        "sentence",
        "technique_id",
        "bow_rank"
    ]
].head(10)

,sentence,technique_id,bow_rank
0,It spreads to Microsoft Windows machines using...,T1210,29
1,"Once a machine is infected by NotPetya, a seri...",T1570,52
2,Ransomware DLL C:\windows\perfc.dat,T1140,122
3,The malware decompresses its resource named 0x...,T1140,42
4,The malware is a DLL that is launched using ru...,T1218.011,1
5,It is executed with the following arguments: #...,T1057,123
6,The credential theft module is dropped if the ...,T1106,93
7,The malware decompresses a resource (0x1 if th...,T1140,17
8,It calls OpenProcess on lsass.exe with access ...,T1003.001,1
9,"Server service, which both enforces security p...",T1003.001,2


In [25]:
#calculate accuracies
recall_at_1 = (
    tram_df["bow_rank"] <= 1
).mean()

recall_at_5 = (
    tram_df["bow_rank"] <= 5
).mean()

recall_at_10 = (
    tram_df["bow_rank"] <= 10
).mean()

In [27]:
# also MRR
mrr = (
    1 / tram_df["bow_rank"]
).mean()

In [28]:
print(f"Recall@1:  {recall_at_1:.3f}")
print(f"Recall@5:  {recall_at_5:.3f}")
print(f"Recall@10: {recall_at_10:.3f}")
print(f"MRR:       {mrr:.3f}")

Recall@1:  0.100
Recall@5:  0.244
Recall@10: 0.324
MRR:       0.177


**NOTES**

TF-IDF results:

- Recall@1  = 0.129
- Recall@5  = 0.305
- Recall@10 = 0.399
- MRR= 0.221

In [29]:
#save result
bow_summary = pd.DataFrame({
    "method": ["Bag-of-Words"],
    "recall@1": [recall_at_1],
    "recall@5": [recall_at_5],
    "recall@10": [recall_at_10],
    "mrr": [mrr]
})

bow_summary

,method,recall@1,recall@5,recall@10,mrr
0,Bag-of-Words,0.100422,0.244394,0.324017,0.177347


In [30]:
bow_summary.to_csv(
    "bow_summary.csv",
    index=False
)

print("BoW summary saved!")

BoW summary saved!


In [31]:
#save query resuls
bow_query_results = tram_df[
    [
        "sentence",
        "technique_id",
        "bow_rank",
        "split"
    ]
].copy()

bow_query_results.to_csv(
    "bow_query_results.csv",
    index=False
)

print("BoW query results saved!")

BoW query results saved!


- save query results so later on we can compare TF-IDF, Bow, and embedding